In [ ]:
# --- PocketLM setup (works in Colab and locally) ---
try:
    import pocketlm  # already installed locally / in CI
except ImportError:
    # Colab: install straight from GitHub. The corpus ships *inside* the package,
    # so there is no repo to clone and no working directory to change.
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "git+https://github.com/ni5h4nt/pocketlm"])
    import pocketlm  # noqa: F811

import os
import torch

DEVICE = "micro" if os.environ.get("POCKETLM_CI") else ("cuda" if torch.cuda.is_available() else "cpu")
CORPUS_PATH = pocketlm.corpus_path()   # packaged data: valid from any directory
print("device:", DEVICE, "(timing is guidance, not a contract)")

# l2.1 Characters as tokens

The simplest tokenizer maps each character to an integer. It is easy to read
and never needs training, which is why PocketLM starts here.

In [ ]:
from pocketlm import CharTokenizer

corpus = open(CORPUS_PATH, encoding="utf-8").read()
tok = CharTokenizer.from_text(corpus)
print("vocab size:", tok.vocab_size)
ids = tok.encode("To be")
print(ids, "->", repr(tok.decode(ids)))

For in-vocab text the round trip is exact. The cost is long sequences (one
token per character) and out-of-vocab holes, which the next lesson fixes.

In [ ]:
assert tok.decode(tok.encode("To be")) == "To be"